# AI Agent Patterns - 6가지 에이전트 설계 패턴

이 노트북에서는 AI 에이전트 구축에 사용되는 6가지 핵심 패턴을 Python 코드로 구현합니다.

## 목차
1. **Prompt Chaining** - 순차적 프롬프트 연결
2. **Routing** - 의도 기반 라우팅
3. **Parallelization** - 병렬 처리
4. **Orchestrator-Workers** - 오케스트레이터-워커 패턴
5. **Evaluator-Optimizer** - 평가-최적화 루프
6. **Autonomous Agent** - 자율 에이전트

## API Key 발급 : https://aistudio.google.com/app/api-keys

## 환경 설정

In [1]:
# 필요한 패키지 설치
!pip install openai anthropic google-genai asyncio  python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 957.0/957.0 kB 16.7 MB/s eta 0:00:00


In [3]:
import os
import json
import asyncio
from typing import List, Dict, Any, Callable
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor

# 데모용 Mock LLM 클래스
class MockLLM:
    """실제 API 호출 없이 패턴을 시연하기 위한 Mock LLM"""

    def __init__(self, name: str = "MockLLM"):
        self.name = name

    def chat(self, prompt: str, system: str = None) -> str:
        """Mock 응답 생성"""
        return f"[{self.name}] 응답: '{prompt[:50]}...'에 대한 처리 완료"

    async def chat_async(self, prompt: str, system: str = None) -> str:
        """비동기 Mock 응답"""
        await asyncio.sleep(0.1)  # API 호출 시뮬레이션
        return self.chat(prompt, system)

# 기본 LLM 인스턴스
llm = MockLLM()

## Google Gemini LLM (무료)

### 무료 등급 모델 (2026년 7월 2일, 실제 API 호출로 검증)

| 모델 | RPM | TPM | RPD | 비고 |
|------|-----|-----|-----|------|
| `gemini-2.5-flash-lite` | 15 | 250K | 1,000 | **추천** (무료 최다 호출) |
| `gemini-2.5-flash` | 10 | 250K | 250 | 균형 |
| `gemini-3-flash-preview` | - | - | - | 최신, 무료 사용 가능 |
| `gemini-3.5-flash` | - | - | - | 최신, 무료 사용 가능 |

모든 무료 모델 1M 토큰 컨텍스트 지원. 할당량은 Google Cloud 프로젝트 단위로 공유되며(API 키를 여러 개 발급해도 합산되지 않음) PT(태평양 시간) 자정에 리셋됩니다. 429 에러는 RPM/TPM/RPD 중 하나만 초과해도 발생합니다.

> **무료 미지원(2026-07-02 실측, `free_tier ... limit: 0`)**: `gemini-2.5-pro`, `gemini-2.0-flash`, `gemini-3-pro-preview`, `gemini-3.1-pro-preview` — Pro 계열과 2.0-flash는 첫 호출부터 429가 나며 유료 등급에서만 사용할 수 있습니다.

> Flash-lite/Flash의 RPM·TPM·RPD 수치는 서드파티 문서 기준이며, 3.x Flash 계열의 정확한 한도는 미공개입니다. 실제 한도는 [AI Studio](https://aistudio.google.com/rate-limit)에서 확인하세요.

| 항목 | 설명 |
|------|------|
| **API 키 발급** | [aistudio.google.com](https://aistudio.google.com) |
| **패키지** | `google-genai` (신규) / `google-generativeai` (deprecated) |

### 참고 링크
- [Gemini API Rate Limits (공식 문서)](https://ai.google.dev/gemini-api/docs/rate-limits)
- [Gemini API Free Tier Rate Limits Guide](https://www.aifreeapi.com/en/posts/gemini-api-free-tier-rate-limits)
- [Gemini API Free Tier Complete Guide](https://www.aifreeapi.com/en/posts/gemini-api-free-tier-complete-guide)

### API 키 설정 모범 사례

**1. 환경 변수 사용 (권장)**

| OS | 터미널 명령어 | 영구 설정 파일 |
|:---|:-------------|:--------------|
| **macOS/Linux** | `export GOOGLE_API_KEY="your-key"` | `~/.zshrc` 또는 `~/.bashrc` |
| **Windows (CMD)** | `set GOOGLE_API_KEY=your-key` | 시스템 환경 변수 설정 |
| **Windows (PowerShell)** | `$env:GOOGLE_API_KEY="your-key"` | PowerShell 프로필 |

```bash
# macOS/Linux - 영구 설정
echo 'export GOOGLE_API_KEY="your-key"' >> ~/.zshrc
source ~/.zshrc
```

```powershell
# Windows PowerShell - 영구 설정
[System.Environment]::SetEnvironmentVariable('GOOGLE_API_KEY','your-key','User')
```

**2. .env 파일 사용 (프로젝트별 관리)**
```bash
# .env 파일 생성
GOOGLE_API_KEY=your-api-key-here
```
```python
# Python에서 로드
from dotenv import load_dotenv
load_dotenv()
```

**3. 보안 주의사항**
- API 키를 코드에 직접 하드코딩하지 마세요
- `.env` 파일은 반드시 `.gitignore`에 추가하세요
- API 키가 노출되면 즉시 [Google AI Studio](https://aistudio.google.com)에서 재발급하세요

In [10]:
from google import genai
import os

class GeminiLLM:
    def __init__(self, model: str = "gemini-3.5-flash", api_key: str = None):
        self.client = genai.Client(api_key=api_key or os.environ.get("GOOGLE_API_KEY"))
        self.model_name = model

    def chat(self, prompt: str, system: str = None) -> str:
        if system:
            full_prompt = f"{system}\n\n{prompt}"
        else:
            full_prompt = prompt

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=full_prompt
        )
        return response.text

    async def chat_async(self, prompt: str, system: str = None) -> str:
        if system:
            full_prompt = f"{system}\n\n{prompt}"
        else:
            full_prompt = prompt

        response = await self.client.aio.models.generate_content(
            model=self.model_name,
            contents=full_prompt
        )
        return response.text

In [ ]:
# API 키 설정
GOOGLE_API_KEY = "AIzaSyBC3-----nipfOCXIJx7ik"

# LLM 인스턴스 생성
llm = GeminiLLM(model="gemini-2.5-flash", api_key=GOOGLE_API_KEY)

# 테스트
print("GeminiLLM 준비 완료!")
response = llm.chat("안녕하세요! 간단히 인사해주세요.")
print(response)


GeminiLLM 준비 완료!
안녕하세요! 반갑습니다. 오늘 어떤 도움이 필요하신가요? 무엇이든 편하게 물어보세요! 😊


---
## Pattern 1: Prompt Chaining (프롬프트 체이닝)

**핵심 개념**: 복잡한 작업을 여러 단계로 분해하여 순차적으로 처리합니다. 각 단계의 출력이 다음 단계의 입력이 됩니다.

**사용 시기**:
- 작업이 명확한 순서로 분해 가능할 때
- 각 단계에서 품질 검증이 필요할 때
- 중간 결과물의 검토가 필요할 때

### 동작 흐름
1. 체인에 단계(name, prompt_template, gate_fn)를 순서대로 등록
2. 초기 입력을 첫 단계 프롬프트의 `{previous_output}` 자리에 주입하여 LLM 호출
3. 게이트 함수로 응답을 검증하고 실패 시 조기 종료
4. 통과한 응답을 다음 단계의 입력으로 넘기며 반복
5. 마지막 단계까지 성공하면 누적된 단계별 결과와 최종 출력을 반환

### 실무 팁
- 단계가 많을수록 지연과 토큰 비용이 선형으로 누적되므로 꼭 필요한 분해만 수행
- 게이트 함수는 JSON 스키마/정규식 등 저렴한 검증을 우선 두고 LLM 재평가는 최소화
- 단일 프롬프트로 충분한 단순 작업에는 오버엔지니어링이 되기 쉬움

In [13]:
class PromptChain:
    """
    Pattern 1: Prompt Chaining
    순차적으로 프롬프트를 연결하여 복잡한 작업을 단계별로 처리
    """

    def __init__(self, llm):
        self.llm = llm
        self.steps = []
        self.results = []

    def add_step(self, name: str, prompt_template: str, gate_fn: Callable = None):
        """
        체인에 단계 추가

        Args:
            name: 단계 이름
            prompt_template: 프롬프트 템플릿 ({previous_output} 사용 가능)
            gate_fn: 다음 단계로 진행 여부를 결정하는 게이트 함수
        """
        # 게이트가 없으면 항상 통과하는 기본 함수를 주입 (실행 루프를 단순화)
        self.steps.append({
            'name': name,
            'prompt_template': prompt_template,
            'gate_fn': gate_fn or (lambda x: True)
        })
        return self

    def run(self, initial_input: str) -> Dict[str, Any]:
        """
        체인 실행
        """
        current_output = initial_input
        self.results = []

        for i, step in enumerate(self.steps):
            print(f"\n{'='*50}")
            print(f"Step {i+1}: {step['name']}")
            print(f"{'='*50}")

            # 1) 프롬프트 생성: 이전 단계 출력을 템플릿에 주입
            prompt = step['prompt_template'].format(previous_output=current_output)
            print(f"Prompt: {prompt[:100]}...")

            # 2) LLM 호출
            response = self.llm.chat(prompt)
            print(f"Response: {response}")

            # 3) 게이트 검증: 실패 시 이후 단계 진행을 막고 조기 반환
            if not step['gate_fn'](response):
                print(f"Gate check failed at step '{step['name']}'")
                return {
                    'success': False,
                    'failed_step': step['name'],
                    'results': self.results
                }

            # 4) 단계 결과 누적 및 다음 입력으로 전달
            self.results.append({
                'step': step['name'],
                'output': response
            })
            current_output = response

        return {
            'success': True,
            'final_output': current_output,
            'results': self.results
        }

In [14]:
# 예시: 마케팅 이메일 작성 체인

# 체인 인스턴스 생성
email_chain = PromptChain(llm)

# 단계 정의: 분석 -> 메시지 도출 -> 초안 -> 톤 조정 순으로 연결
email_chain.add_step(
    name="고객 분석",
    prompt_template="""당신은 B2C/B2B SaaS 마케팅 전략가입니다.
아래 고객 프로필을 근거로 핵심 니즈, 예상 페인 포인트, 관심 키워드, 선호 커뮤니케이션 톤을 각 1문장으로 정리하세요.
출력은 JSON 형식으로 keys: needs, pain_points, keywords, tone.

[고객 프로필]
{previous_output}"""
).add_step(
    name="핵심 메시지 도출",
    prompt_template="""위 분석 결과를 바탕으로 이메일 1통에 담을 메시지를 설계하세요.
- 한 줄 가치 제안(Value Proposition) 1개
- 이를 뒷받침하는 근거(Proof Point) 3개
- 행동 유도 문구(CTA) 후보 2개
출력은 Markdown 불릿 리스트.

[분석 결과]
{previous_output}"""
).add_step(
    name="이메일 초안 작성",
    prompt_template="""위 메시지 설계를 바탕으로 이메일 초안을 작성하세요.
제약:
- 제목 45자 이내, 개인화 키워드 1개 이상 포함
- 본문 120~180단어, 단락 3개
- 말미에 CTA 버튼 문구와 {{cta_link}} placeholder 포함
출력은 'Subject:'와 'Body:' 두 섹션으로 구분된 Plain Text.

[메시지 설계]
{previous_output}"""
).add_step(
    name="톤 조정 및 최종화",
    prompt_template="""아래 초안을 30대 IT 종사자에게 자연스럽게 읽히도록 수정하세요.
조정 기준:
- 업계 클리셰 및 번역투 제거
- 기술 용어 정확성 유지
- 친근하되 가볍지 않은 톤
최종 버전만 출력하고 수정 사유는 쓰지 마세요.

[초안]
{previous_output}"""
)

# 체인 실행: 초기 입력이 첫 단계의 {previous_output}에 주입
result = email_chain.run(
    "고객: 32세 스타트업 백엔드 개발자. "
    "관심사: LLM 기반 개발 생산성, 멀티 클라우드 비용 최적화. "
    "페인 포인트: 반복 보일러플레이트 작성, 인프라 비용 증가."
)
print(f"\n최종 결과: {result['success']}")


Step 1: 고객 분석
Prompt: 당신은 B2C/B2B SaaS 마케팅 전략가입니다.
아래 고객 프로필을 근거로 핵심 니즈, 예상 페인 포인트, 관심 키워드, 선호 커뮤니케이션 톤을 각 1문장으로 정리하세요.
출력...
Response: ```json
{
  "needs": "LLM을 활용해 개발 프로세스를 자동화하여 생산성을 높이고, 효율적인 자원 관리로 멀티 클라우드 인프라 비용을 최적화하고자 합니다.",
  "pain_points": "단순 반복적인 보일러플레이트 코드 작성에 많은 리소스를 낭비하고 있으며, 제어하기 어려울 정도로 증가하는 멀티 클라우드 인프라 비용에 부담을 느끼고 있습니다.",
  "keywords": "LLM 기반 코드 생성, 개발 생산성 도구, 멀티 클라우드 비용 최적화, FinOps, 보일러플레이트 자동화 솔루션",
  "tone": "불필요한 수식어를 배제하고 구체적인 기술적 효용과 정량적인 비용 절감 데이터를 명확히 제시하는 전문적이고 실용적인 톤앤매너를 선호합니다."
}
```

Step 2: 핵심 메시지 도출
Prompt: 위 분석 결과를 바탕으로 이메일 1통에 담을 메시지를 설계하세요.
- 한 줄 가치 제안(Value Proposition) 1개
- 이를 뒷받침하는 근거(Proof Point) 3개...
Response: 제공해주신 분석 결과를 바탕으로, 불필요한 미사여구를 배제하고 정량적 효과와 기술적 효용을 강조한 이메일 메시지를 설계했습니다.

*   **한 줄 가치 제안 (Value Proposition)**
    *   "LLM 기반의 개발 프로세스 자동화로 보일러플레이트 코드를 제거하고, 정밀한 FinOps 관리로 멀티 클라우드 인프라 비용을 최적화하십시오."

*   **뒷받침하는 근거 (Proof Points)**
    *   **반복 개발 리소스 80% 감축:** LLM 기반 코드 생성 엔진을 통해 단순 반복적인 보일러플레이트 코드 작성 시간을 최소화하여, 개발자가 핵심 비즈니스 로직 구현에만 집

---
## Pattern 2: Routing (라우팅)

**핵심 개념**: 입력을 분류하여 적절한 처리 경로로 전달합니다. 각 경로는 특화된 프롬프트나 모델을 사용합니다.

**사용 시기**:
- 입력 유형에 따라 다른 처리가 필요할 때
- 전문화된 응답이 필요할 때
- 비용/성능 최적화가 필요할 때

### 동작 흐름
1. 각 의도(intent)에 대해 핸들러와 설명을 등록
2. 입력이 들어오면 LLM 또는 규칙 기반으로 의도를 분류
3. 분류된 의도에 매핑된 핸들러로 디스패치
4. 매칭되는 의도가 없으면 default 핸들러로 폴백
5. 실행 결과와 분류된 intent를 함께 반환

### 실무 팁
- 분류기 자체의 오류가 전체 품질의 상한이 되므로 분류 정확도 모니터링이 필수
- 경로가 10개 이상이면 단일 분류기보다 계층적 분류가 안정적
- 대부분의 입력이 한 경로로 쏠리면 라우팅이 아닌 단일 프롬프트로 충분할 수 있음

In [ ]:
class Router:
    """
    Pattern 2: Routing
    입력 의도를 분석하여 적절한 처리 경로로 라우팅
    """

    def __init__(self, llm):
        self.llm = llm
        self.routes = {}
        self.default_route = None

    def add_route(self, intent: str, handler: Callable, description: str = ""):
        """
        라우팅 경로 추가

        Args:
            intent: 의도 이름
            handler: 해당 의도 처리 함수
            description: 의도 설명 (분류에 사용)
        """
        self.routes[intent] = {
            'handler': handler,
            'description': description
        }
        return self

    def set_default(self, handler: Callable):
        """기본 처리 경로 설정"""
        self.default_route = handler
        return self

    def classify_intent(self, user_input: str) -> str:
        """
        LLM을 사용하여 입력 의도 분류
        """
        # 1) 등록된 경로 설명을 모아 분류용 프롬프트 구성
        route_descriptions = "\n".join([
            f"- {intent}: {info['description']}"
            for intent, info in self.routes.items()
        ])

        classification_prompt = f"""다음 사용자 입력의 의도를 분류하세요.

가능한 의도:
{route_descriptions}

사용자 입력: {user_input}

의도 (위 목록 중 하나만 출력):"""

        # 2) 데모: 키워드 매칭 기반 분류 (실제 환경에서는 LLM 응답 파싱으로 대체)
        input_lower = user_input.lower()
        for intent in self.routes.keys():
            if intent.lower() in input_lower:
                return intent

        # LLM 기반 분류 (실제 구현)
        # response = self.llm.chat(classification_prompt)
        # return response.strip()

        # 3) 매칭 실패 시 첫 번째 경로로 폴백 (등록 경로가 없으면 unknown)
        return list(self.routes.keys())[0] if self.routes else "unknown"

    def route(self, user_input: str) -> Dict[str, Any]:
        """
        입력을 분류하고 적절한 핸들러로 라우팅
        """
        # 1) 의도 분류
        intent = self.classify_intent(user_input)
        print(f"분류된 의도: {intent}")

        # 2) 의도에 매핑된 핸들러 실행, 없으면 default → 그마저 없으면 거부 응답
        if intent in self.routes:
            handler = self.routes[intent]['handler']
            result = handler(user_input)
        elif self.default_route:
            print(f"알 수 없는 의도, 기본 경로 사용")
            result = self.default_route(user_input)
        else:
            result = "처리할 수 없는 요청입니다."

        return {
            'intent': intent,
            'result': result
        }

In [ ]:
# 예시: 고객 서비스 라우터

# 경로별 핸들러 정의 (실제로는 각 팀의 전문 프롬프트/모델로 교체)
def handle_billing(query: str) -> str:
    return f"[결제팀] 결제 관련 문의 처리: {query}"

def handle_technical(query: str) -> str:
    return f"[기술팀] 기술 지원 처리: {query}"

def handle_sales(query: str) -> str:
    return f"[영업팀] 영업 문의 처리: {query}"

def handle_general(query: str) -> str:
    return f"[일반] 일반 문의 처리: {query}"

# 라우터 설정: 의도 이름 + 핸들러 + 분류용 설명(LLM/규칙 분류기의 힌트)
support_router = Router(llm)

support_router.add_route(
    "billing", handle_billing,
    "청구서/결제 실패/환불/카드 변경/영수증/미납금 관련 문의"
).add_route(
    "technical", handle_technical,
    "로그인 실패/에러 메시지/앱 크래시/성능 저하/API 오류/버그 제보 관련 문의"
).add_route(
    "sales", handle_sales,
    "플랜 비교/엔터프라이즈 견적/볼륨 할인/계약 조건/무료 체험 연장 관련 문의"
).set_default(handle_general)

# 테스트: 입력별로 분류된 의도와 핸들러 결과를 확인
queries = [
    # "billing 관련 문의: 지난달 청구서 금액이 예상보다 2배 많이 청구되었습니다",
    # "technical 이슈: 모바일 앱이 로그인 직후 즉시 크래시되고 로그에 NullPointerException이 찍힙니다",
    "sales 문의: 팀 규모 50명 엔터프라이즈 플랜 볼륨 할인과 연간 계약 조건을 알고 싶습니다"
]

for query in queries:
    print(f"\n입력: {query}")
    result = support_router.route(query)
    print(f"결과: {result['result']}")


입력: sales 문의: 팀 규모 50명 엔터프라이즈 플랜 볼륨 할인과 연간 계약 조건을 알고 싶습니다
분류된 의도: sales
결과: [영업팀] 영업 문의 처리: sales 문의: 팀 규모 50명 엔터프라이즈 플랜 볼륨 할인과 연간 계약 조건을 알고 싶습니다


---
## Pattern 3: Parallelization (병렬화)

**핵심 개념**: 독립적인 작업을 동시에 처리하여 효율성을 높입니다.

**두 가지 방식**:
1. **Sectioning**: 하나의 작업을 독립적인 부분으로 나누어 병렬 처리
2. **Voting**: 동일한 작업을 여러 번 수행하고 결과를 집계

### 동작 흐름
1. 작업을 독립적인 서브태스크 목록 또는 동일 프롬프트 N회로 준비
2. `asyncio.gather`로 모든 LLM 호출을 동시에 실행
3. 모든 응답이 도착할 때까지 대기
4. Sectioning은 이름별로 결과를 묶고, Voting은 aggregator로 합의 결과 도출
5. 통합된 결과를 반환

### 실무 팁
- 작업 간 의존성이 있으면 병렬화 대신 Chaining이나 Orchestrator 패턴이 적합
- 동시 호출은 레이트리밋과 비용을 빠르게 소모하므로 세마포어/배치 크기 제어가 필요
- Voting은 정답이 수렴하는 문제에만 효과적이며 창의적 생성에는 오히려 역효과

In [ ]:
class ParallelProcessor:
    """
    Pattern 3: Parallelization
    독립적인 작업을 병렬로 처리
    """

    def __init__(self, llm):
        self.llm = llm

    async def section_parallel(self, tasks: List[Dict[str, str]]) -> List[Dict[str, Any]]:
        """
        Sectioning: 여러 독립적인 작업을 병렬 처리

        Args:
            tasks: [{'name': 'task_name', 'prompt': 'prompt_text'}, ...]
        """
        print(f"Sectioning: {len(tasks)}개 작업 병렬 처리 시작")

        # 각 태스크를 비동기 코루틴으로 래핑하여 결과에 이름을 유지
        async def process_task(task):
            result = await self.llm.chat_async(task['prompt'])
            return {'name': task['name'], 'result': result}

        # gather로 동시 실행 (의존성이 없는 작업만 안전)
        results = await asyncio.gather(*[process_task(t) for t in tasks])

        print(f"모든 작업 완료")
        return list(results)

    async def voting_parallel(self, prompt: str, num_votes: int = 3,
                               aggregator: Callable = None) -> Dict[str, Any]:
        """
        Voting: 동일한 작업을 여러 번 수행하고 결과 집계

        Args:
            prompt: 처리할 프롬프트
            num_votes: 투표 수
            aggregator: 결과 집계 함수
        """
        print(f"Voting: {num_votes}회 병렬 실행 시작")

        # 1) 동일 프롬프트를 num_votes회 병렬 호출 (샘플링 다양성으로 합의 유도)
        tasks = [self.llm.chat_async(prompt) for _ in range(num_votes)]
        responses = await asyncio.gather(*tasks)

        # 2) 집계 전략: 사용자가 주입한 aggregator, 없으면 첫 응답을 대표값으로 사용
        if aggregator:
            final_result = aggregator(list(responses))
        else:
            final_result = responses[0]

        return {
            'all_responses': list(responses),
            'final_result': final_result,
            'num_votes': num_votes
        }

In [ ]:
# 예시 1: Sectioning - 문서 분석의 여러 측면을 병렬 처리

async def demo_sectioning():
    processor = ParallelProcessor(llm)

    document = (
        "2026년 1분기 글로벌 LLM 시장은 전년 대비 62% 성장하며 "
        "기업용 AI 에이전트 플랫폼 채택이 본격화되었다. "
        "오픈소스 모델 점유율은 28%로 상승했으나, 보안/규제 이슈로 "
        "상용 API 선호가 여전히 우세하다. 주요 도입 장벽은 "
        "환각(hallucination) 관리, 도메인 파인튜닝 비용, 데이터 프라이버시다."
    )

    # 서로 독립적인 분석 태스크 4개를 구성 (의존성 없음 -> 병렬 안전)
    analysis_tasks = [
        {
            'name': '감성 분석',
            'prompt': f"""다음 문서의 전반 감성 톤을 분석하세요.
- 전반 감성: positive / neutral / negative 중 택1
- 감정을 드러내는 근거 구절 3개를 원문에서 인용
- 문서 전체 감성이 일관적인지 혼재인지 한 줄 판단

[문서]
{document}"""
        },
        {
            'name': '핵심 키워드',
            'prompt': f"""다음 문서에서 검색/태깅/분류에 유용한 키워드를 추출하세요.
- 핵심 엔티티(기업/기술/제품) 최대 5개
- 도메인 고유 용어 최대 5개
- 일반 명사(예: 시장, 성장) 제외
출력은 JSON 배열 2개 (keys: entities, domain_terms).

[문서]
{document}"""
        },
        {
            'name': '요약',
            'prompt': f"""경영진이 30초 안에 파악할 수 있도록 다음 문서를 3줄로 요약하세요.
- 지표/숫자는 반드시 포함
- 불확실하거나 추정이 섞인 내용은 '~로 추정' 등 완곡 표현 사용

[문서]
{document}"""
        },
        {
            'name': '관련 주제',
            'prompt': f"""아래 문서와 인접하여 후속 조사가 유용할 주제 5개를 제안하세요.
- 각 주제마다 1줄 이유 첨부
- 문서에 이미 다뤄진 주제는 제외
출력은 Markdown 불릿.

[문서]
{document}"""
        }
    ]

    # 네 개 호출을 동시에 실행하고 결과를 이름별로 수집
    results = await processor.section_parallel(analysis_tasks)

    print("\n== 분석 결과 ==")
    for r in results:
        print(f"  - {r['name']}: {r['result']}")

    return results

# Jupyter에서 실행
await demo_sectioning()

Sectioning: 4개 작업 병렬 처리 시작


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 8.199483423s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flas

In [ ]:
# 예시 2: Voting - 코드 리뷰 결과 집계

async def demo_voting():
    processor = ParallelProcessor(llm)

    # 동일 프롬프트를 여러 번 돌려 합의 결과를 얻음 (SQL 인젝션 취약 코드 예시)
    code_review_prompt = """당신은 보안 중심 코드 리뷰어입니다.
아래 Python 함수를 OWASP Top 10 관점에서 검토하고 다음 형식으로 답하세요.

- 탐지 취약점: 이름 / CWE 번호 / 심각도(Critical|High|Medium|Low)
- 공격 시나리오: 1~2문장
- 수정 코드: parameterized query로 리팩터링한 스니펫
- 추가 권고: 1가지

[코드]
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return db.execute(query)
"""

    # 집계 전략: 여러 응답 중 다수 의견을 최종 결과로 채택
    def majority_vote(responses):
        """다수결로 결과 결정"""
        return f"[다수결 결과] {len(responses)}개 응답 중 분석 완료"

    # 3회 병렬 실행 후 majority_vote로 합의
    result = await processor.voting_parallel(
        prompt=code_review_prompt,
        num_votes=3,
        aggregator=majority_vote
    )

    print(f"\n== 투표 결과 ==")
    print(f"  총 응답 수: {result['num_votes']}")
    print(f"  최종 결과: {result['final_result']}")

    return result

await demo_voting()

Voting: 3회 병렬 실행 시작

== 투표 결과 ==
  총 응답 수: 3
  최종 결과: [다수결 결과] 3개 응답 중 분석 완료


{'all_responses': ['## 보안 중심 코드 검토 결과\n\n**함수:** `get_user(user_id)`\n\n### 탐지 취약점\n\n*   **SQL Injection** / CWE-89 / **Critical**\n\n### 공격 시나리오\n\n공격자는 `user_id` 값에 악의적인 SQL 코드를 삽입하여 데이터베이스의 데이터를 탈취하거나 조작할 수 있습니다. 예를 들어, `user_id`를 `\' OR \'1\'=\'1`와 같이 전달하면 모든 사용자 정보를 조회할 수 있습니다.\n\n### 수정 코드 (Parameterized Query)\n\n```python\ndef get_user(user_id):\n    query = "SELECT * FROM users WHERE id = ?"  # 또는 %s, :user_id 등 DB 라이브러리에 따라 다름\n    return db.execute(query, (user_id,)) # 튜플 형태로 전달\n```\n\n**설명:**\n\n위 코드에서는 `f-string`을 사용하여 `user_id`를 직접 쿼리 문자열에 포함시키는 대신, 플레이스홀더(`?` 또는 DB 라이브러리에 따라 `%s`, `:user_id` 등)를 사용하고 `db.execute` 함수의 두 번째 인자로 `user_id`를 별도의 튜플 형태로 전달합니다. 이렇게 하면 데이터베이스 드라이버가 `user_id`를 SQL 코드의 일부가 아닌 데이터 값으로 안전하게 처리하여 SQL Injection 공격을 방지합니다.\n\n### 추가 권고\n\n*   **입력값 검증:** `user_id`가 예상되는 데이터 타입(예: 정수)인지, 유효한 범위 내의 값인지 등을 함수 시작 부분에서 검증하는 것이 좋습니다. 이를 통해 악의적인 입력값뿐만 아니라 의도치 않은 잘못된 입력으로 인한 문제를 사전에 방지할 수 있습니다.',
  '## 보안 중심 코드 리뷰\n\n**탐지 취약점:** SQL Injection / CWE-89 / Crit

---
## Pattern 4: Orchestrator-Workers (오케스트레이터-워커)

**핵심 개념**: 중앙 오케스트레이터가 작업을 분석하고 전문화된 워커에게 하위 작업을 분배합니다.

**사용 시기**:
- 복잡한 작업을 동적으로 분해해야 할 때
- 전문화된 처리가 필요한 하위 작업이 있을 때
- 작업 구조를 미리 예측하기 어려울 때

### 동작 흐름
1. 오케스트레이터가 상위 작업을 받아 하위 작업(SubTask) 목록으로 분해
2. 각 SubTask의 worker_type과 dependencies를 함께 결정
3. 의존성이 해소된 SubTask부터 해당 워커로 디스패치하여 실행
4. 실행 결과를 task_results에 누적하고 후속 워커의 컨텍스트로 전달
5. 모든 SubTask가 완료되면 오케스트레이터가 결과를 통합하여 반환

### 실무 팁
- 분해 단계 자체가 실패하면 이후 전체가 무너지므로 분해 결과 검증/재시도 로직이 필요
- 워커가 너무 세분화되면 오버헤드와 컨텍스트 전달 비용이 증가
- 정적 워크플로우로 충분하다면 Chaining을 쓰는 편이 단순하고 예측 가능

In [ ]:
@dataclass
class SubTask:
    """하위 작업 정의"""
    id: str
    description: str
    worker_type: str
    dependencies: List[str] = None

    def __post_init__(self):
        # dataclass 기본값으로 가변 리스트를 직접 쓰면 공유되므로 여기서 초기화
        if self.dependencies is None:
            self.dependencies = []


class OrchestratorWorkers:
    """
    Pattern 4: Orchestrator-Workers
    중앙 오케스트레이터가 작업을 분해하고 워커에게 분배
    """

    def __init__(self, llm):
        self.llm = llm
        self.workers = {}
        self.task_results = {}

    def register_worker(self, worker_type: str, handler: Callable):
        """워커 등록"""
        self.workers[worker_type] = handler
        return self

    def decompose_task(self, task: str) -> List[SubTask]:
        """
        오케스트레이터: 작업을 하위 작업으로 분해
        실제 구현에서는 LLM을 사용하여 동적으로 분해
        """
        print(f"\n오케스트레이터: 작업 분석 중...")
        print(f"   원본 작업: {task}")

        # 데모: 고정된 하위 작업 DAG (실제로는 LLM이 task를 보고 동적으로 생성)
        subtasks = [
            SubTask("1", "요구사항 분석", "analyst"),
            SubTask("2", "아키텍처 설계", "architect", ["1"]),
            SubTask("3", "코드 구현", "developer", ["2"]),
            SubTask("4", "테스트 작성", "tester", ["3"]),
            SubTask("5", "문서화", "writer", ["3"])
        ]

        print(f"   분해된 하위 작업: {len(subtasks)}개")
        for st in subtasks:
            deps = f" (의존: {st.dependencies})" if st.dependencies else ""
            print(f"      - [{st.id}] {st.description} → {st.worker_type}{deps}")

        return subtasks

    def execute_subtask(self, subtask: SubTask) -> str:
        """하위 작업 실행"""
        if subtask.worker_type in self.workers:
            # 의존 태스크 결과를 컨텍스트로 전달하여 연계 정보 활용
            context = {dep: self.task_results.get(dep) for dep in subtask.dependencies}
            return self.workers[subtask.worker_type](subtask.description, context)
        else:
            return f"[기본 처리] {subtask.description}"

    def run(self, task: str) -> Dict[str, Any]:
        """
        전체 워크플로우 실행
        """
        # 1) 작업 분해
        subtasks = self.decompose_task(task)

        # 2) 의존성이 해소된 순서대로 워커 실행
        print(f"\n워커 실행 시작...")
        self.task_results = {}

        executed = set()
        while len(executed) < len(subtasks):
            for subtask in subtasks:
                if subtask.id in executed:
                    continue

                # 모든 선행 태스크가 완료되었을 때만 실행 (순진한 토폴로지 순서)
                if all(dep in executed for dep in subtask.dependencies):
                    print(f"   [{subtask.worker_type}] {subtask.description} 실행")
                    result = self.execute_subtask(subtask)
                    self.task_results[subtask.id] = result
                    executed.add(subtask.id)
                    print(f"      완료: {result[:50]}...")

        # 3) 결과 통합 (오케스트레이터의 최종 요약 단계)
        print(f"\n오케스트레이터: 결과 통합 중...")

        return {
            'task': task,
            'subtasks': len(subtasks),
            'results': self.task_results
        }

In [ ]:
# 예시: 소프트웨어 개발 오케스트레이터

# 역할별 워커 함수 정의 (실제로는 각자 다른 프롬프트/모델을 사용)
def analyst_worker(task: str, context: Dict) -> str:
    return f"[분석 완료] {task}: 기능 요구사항 3개, 비기능 요구사항 2개 도출"

def architect_worker(task: str, context: Dict) -> str:
    return f"[설계 완료] {task}: 마이크로서비스 아키텍처, 3개 서비스 구성"

def developer_worker(task: str, context: Dict) -> str:
    return f"[개발 완료] {task}: Python 코드 500줄 작성"

def tester_worker(task: str, context: Dict) -> str:
    return f"[테스트 완료] {task}: 단위 테스트 20개, 통합 테스트 5개 작성"

def writer_worker(task: str, context: Dict) -> str:
    return f"[문서화 완료] {task}: API 문서 및 사용자 가이드 작성"

# 오케스트레이터 설정: worker_type 이름으로 핸들러 매핑
orchestrator = OrchestratorWorkers(llm)

orchestrator.register_worker("analyst", analyst_worker)
orchestrator.register_worker("architect", architect_worker)
orchestrator.register_worker("developer", developer_worker)
orchestrator.register_worker("tester", tester_worker)
orchestrator.register_worker("writer", writer_worker)

# 실행: 상위 작업을 넘기면 내부에서 분해 → 의존성 순서대로 워커 호출
result = orchestrator.run("사용자 인증 시스템 개발")
print(f"\n최종 결과: {result['subtasks']}개 하위 작업 완료")


오케스트레이터: 작업 분석 중...
   원본 작업: 사용자 인증 시스템 개발
   분해된 하위 작업: 5개
      - [1] 요구사항 분석 → analyst
      - [2] 아키텍처 설계 → architect (의존: ['1'])
      - [3] 코드 구현 → developer (의존: ['2'])
      - [4] 테스트 작성 → tester (의존: ['3'])
      - [5] 문서화 → writer (의존: ['3'])

워커 실행 시작...
   [analyst] 요구사항 분석 실행
      완료: [분석 완료] 요구사항 분석: 기능 요구사항 3개, 비기능 요구사항 2개 도출...
   [architect] 아키텍처 설계 실행
      완료: [설계 완료] 아키텍처 설계: 마이크로서비스 아키텍처, 3개 서비스 구성...
   [developer] 코드 구현 실행
      완료: [개발 완료] 코드 구현: Python 코드 500줄 작성...
   [tester] 테스트 작성 실행
      완료: [테스트 완료] 테스트 작성: 단위 테스트 20개, 통합 테스트 5개 작성...
   [writer] 문서화 실행
      완료: [문서화 완료] 문서화: API 문서 및 사용자 가이드 작성...

오케스트레이터: 결과 통합 중...

최종 결과: 5개 하위 작업 완료


---
## Pattern 5: Evaluator-Optimizer (평가-최적화)

**핵심 개념**: Generator가 결과물을 생성하고, Evaluator가 품질을 평가합니다. 기준 미달 시 피드백과 함께 재생성을 요청합니다.

**사용 시기**:
- 명확한 품질 기준이 존재할 때
- 반복적 개선이 가치를 더할 때
- 자동화된 품질 보증이 필요할 때

### 동작 흐름
1. Generator가 작업 프롬프트(+이전 피드백)로 초안을 생성
2. Evaluator가 기준(criteria)에 따라 점수와 피드백을 산출
3. 점수가 pass_threshold 이상이면 성공으로 종료
4. 기준 미달 시 피드백을 다음 iteration의 Generator 입력으로 전달
5. max_iterations 도달 시 마지막 결과와 이력을 반환

### 실무 팁
- 평가 기준이 주관적이면 루프가 수렴하지 않아 토큰만 소모됨
- Evaluator와 Generator에 같은 모델을 쓰면 같은 실수를 통과시키기 쉬움 — 다른 모델/규칙을 혼용
- 임계값을 너무 높이면 무한 루프에 가까워지므로 max_iterations로 안전장치 필수

In [ ]:
@dataclass
class EvaluationResult:
    """평가 결과"""
    passed: bool
    score: float
    feedback: str


class EvaluatorOptimizer:
    """
    Pattern 5: Evaluator-Optimizer
    생성-평가-개선 반복 루프
    """

    def __init__(self, llm, max_iterations: int = 5, pass_threshold: float = 80.0):
        self.llm = llm
        self.max_iterations = max_iterations
        self.pass_threshold = pass_threshold
        self.history = []

    def generate(self, prompt: str, feedback: str = None) -> str:
        """
        Generator: 콘텐츠 생성
        """
        # 이전 피드백이 있으면 프롬프트 끝에 덧붙여 개선 방향을 지시
        if feedback:
            full_prompt = f"{prompt}\n\n이전 피드백을 반영하세요: {feedback}"
        else:
            full_prompt = prompt

        return self.llm.chat(full_prompt)

    def evaluate(self, content: str, criteria: List[str]) -> EvaluationResult:
        """
        Evaluator: 품질 평가
        실제 구현에서는 LLM 또는 규칙 기반 평가
        """
        # 데모: 반복이 쌓일수록 점수가 오르는 가짜 채점기 (루프 흐름을 보이기 위함)
        base_score = 60 + len(self.history) * 15
        score = min(base_score, 100)

        passed = score >= self.pass_threshold

        # 실패 시 어떤 기준을 아직 충족 못했는지 피드백으로 되돌려줌
        if not passed:
            feedback = f"점수 {score}/100. 개선 필요: {criteria[len(self.history) % len(criteria)]}"
        else:
            feedback = f"점수 {score}/100. 모든 기준 충족!"

        return EvaluationResult(passed=passed, score=score, feedback=feedback)

    def run(self, task: str, criteria: List[str]) -> Dict[str, Any]:
        """
        평가-최적화 루프 실행
        """
        print(f"\nEvaluator-Optimizer 시작")
        print(f"   작업: {task}")
        print(f"   평가 기준: {criteria}")
        print(f"   통과 기준: {self.pass_threshold}점 이상")
        print(f"   최대 반복: {self.max_iterations}회")

        self.history = []
        current_content = None
        feedback = None

        for iteration in range(1, self.max_iterations + 1):
            print(f"\n--- Iteration {iteration} ---")

            # 1) 생성: 첫 반복은 피드백 없음, 이후는 직전 피드백을 반영
            print(f"Generator: 콘텐츠 생성 중...")
            current_content = self.generate(task, feedback)
            print(f"   생성 결과: {current_content[:50]}...")

            # 2) 평가
            print(f"Evaluator: 품질 평가 중...")
            eval_result = self.evaluate(current_content, criteria)

            # 3) 이력 누적 (감사 로그/후처리에 사용)
            self.history.append({
                'iteration': iteration,
                'content': current_content,
                'score': eval_result.score,
                'passed': eval_result.passed,
                'feedback': eval_result.feedback
            })

            # 4) 통과 시 조기 종료, 실패 시 피드백을 다음 반복으로 전달
            if eval_result.passed:
                print(f"PASS: {eval_result.feedback}")
                return {
                    'success': True,
                    'iterations': iteration,
                    'final_content': current_content,
                    'final_score': eval_result.score,
                    'history': self.history
                }
            else:
                print(f"FAIL: {eval_result.feedback}")
                feedback = eval_result.feedback

        # 5) 안전장치: max_iterations 도달 시 마지막 결과 반환
        print(f"\n최대 반복 횟수 도달")
        return {
            'success': False,
            'iterations': self.max_iterations,
            'final_content': current_content,
            'final_score': self.history[-1]['score'],
            'history': self.history
        }

In [ ]:
# 예시: 마케팅 카피 최적화

# Optimizer 인스턴스: 최대 5회 반복, 80점 이상이면 통과
optimizer = EvaluatorOptimizer(llm, max_iterations=5, pass_threshold=80)

# 평가 기준 (Evaluator가 부족분을 피드백으로 되돌려줌)
criteria = [
    "측정 가능한 가치 제안(수치/구체 결과)이 포함될 것",
    "시니어 엔지니어의 일상 언어 사용, 과장/번역투/클리셰 금지",
    "헤드라인 15단어 이내, 서브헤드라인 30단어 이내",
    "낚시성 없이 클릭 유도성이 유지될 것",
    "차별점(보안 취약점 자동 탐지, 팀 컨벤션 학습)이 명시적 또는 암시적으로 드러날 것"
]

# 실행: 생성 -> 평가 -> 피드백 -> 재생성 루프를 통과하거나 max_iterations까지
result = optimizer.run(
    task=(
        "B2B 개발자 대상 'AI 코드 리뷰어' 제품의 랜딩 페이지 Hero 섹션 "
        "헤드라인 1줄과 서브헤드라인 1~2줄을 작성하세요. "
        "타겟은 시니어 백엔드/플랫폼 엔지니어이며, "
        "경쟁 제품(GitHub Copilot, CodeRabbit) 대비 차별점은 "
        "보안 취약점 자동 탐지와 팀 컨벤션 학습입니다. "
        "모호한 최상급 표현(최고의, 혁신적인)은 사용 금지."
    ),
    criteria=criteria
)

print(f"\n== 최종 결과 ==")
print(f"   성공 여부: {result['success']}")
print(f"   반복 횟수: {result['iterations']}")
print(f"   최종 점수: {result['final_score']}")

---
## Pattern 6: Autonomous Agent (자율 에이전트)

**핵심 개념**: 최소한의 입력(단일 목표)만으로 독립적으로 작동합니다. 에이전트는 스스로 행동을 취하고, 환경 피드백을 관찰하며, 목표 달성 여부를 자체 평가합니다.

**주요 구성 요소**:
- Goal (목표)
- Tools (도구)
- Memory (메모리)
- Self-evaluation (자체 평가)

### 동작 흐름
1. 목표와 사용 가능한 도구 목록을 에이전트에 주입
2. Think 단계에서 메모리와 도구 설명을 바탕으로 다음 행동을 결정
3. Act 단계에서 선택한 도구를 호출하고 결과(Observation)를 수집
4. 행동/결과를 메모리에 기록하고 목표 달성 여부를 self-evaluate
5. 달성되면 종료, 아니면 max_steps까지 Think-Act-Observe 루프 반복

### 실무 팁
- 무한 루프/도구 오남용 위험이 크므로 max_steps, 예산 제한, 화이트리스트 도구가 기본
- 자체 평가는 과신되기 쉬워 외부 검증(규칙/테스트/휴먼)과 병행하는 편이 안전
- 작업이 결정적 워크플로우로 표현 가능하면 덜 자율적인 패턴(Orchestrator 등)이 비용·안정성 면에서 우수

In [ ]:
@dataclass
class AgentAction:
    """에이전트 행동"""
    tool: str
    input: str
    thought: str


@dataclass
class AgentObservation:
    """환경 관찰 결과"""
    result: str
    success: bool


class AutonomousAgent:
    """
    Pattern 6: Autonomous Agent
    목표 기반 자율 에이전트
    """

    def __init__(self, llm, max_steps: int = 10):
        self.llm = llm
        self.max_steps = max_steps
        self.tools = {}
        self.memory = []  # 행동 기록

    def register_tool(self, name: str, func: Callable, description: str):
        """도구 등록"""
        self.tools[name] = {
            'func': func,
            'description': description
        }
        return self

    def think(self, goal: str) -> AgentAction:
        """
        현재 상태를 분석하고 다음 행동 결정
        실제 구현에서는 LLM이 ReAct 패턴으로 추론
        """
        # 1) 기존 메모리를 Step 요약으로 직렬화 → LLM 컨텍스트 입력
        context = "\n".join([
            f"Step {i+1}: {m['action']} → {m['result']}"
            for i, m in enumerate(self.memory)
        ])

        # 2) 도구 설명 리스트 구성 (LLM이 선택지를 인지하도록)
        tool_descriptions = "\n".join([
            f"- {name}: {info['description']}"
            for name, info in self.tools.items()
        ])

        # 3) 데모용 규칙 기반 계획 (실제로는 LLM이 context/tools로 다음 도구 선택)
        if len(self.memory) == 0:
            return AgentAction(
                tool="search",
                input=goal,
                thought="목표 달성을 위해 먼저 정보를 검색해야 합니다."
            )
        elif len(self.memory) == 1:
            return AgentAction(
                tool="analyze",
                input=self.memory[-1]['result'],
                thought="검색 결과를 분석하여 다음 단계를 결정합니다."
            )
        elif len(self.memory) == 2:
            return AgentAction(
                tool="execute",
                input="분석 결과 기반 실행",
                thought="분석 결과를 바탕으로 작업을 실행합니다."
            )
        else:
            # finish는 루프 종료를 명시적으로 선언하는 특수 액션
            return AgentAction(
                tool="finish",
                input="목표 달성 완료",
                thought="모든 단계가 완료되어 작업을 종료합니다."
            )

    def act(self, action: AgentAction) -> AgentObservation:
        """
        행동 실행 및 환경 피드백 수집
        """
        # 등록된 도구면 호출하고 예외는 실패 Observation으로 변환
        if action.tool in self.tools:
            try:
                result = self.tools[action.tool]['func'](action.input)
                return AgentObservation(result=result, success=True)
            except Exception as e:
                return AgentObservation(result=str(e), success=False)
        elif action.tool == "finish":
            return AgentObservation(result="Task completed", success=True)
        else:
            return AgentObservation(result=f"Unknown tool: {action.tool}", success=False)

    def is_goal_achieved(self, goal: str) -> bool:
        """
        목표 달성 여부 자체 평가
        """
        # 데모: 3단계 이상 기록되면 달성으로 판정 (실제로는 LLM이 목표와 대조)
        return len(self.memory) >= 3

    def run(self, goal: str) -> Dict[str, Any]:
        """
        자율 에이전트 실행 루프
        """
        print(f"\nAutonomous Agent 시작")
        print(f"   목표: {goal}")
        print(f"   사용 가능 도구: {list(self.tools.keys())}")
        print(f"   최대 스텝: {self.max_steps}")

        self.memory = []

        for step in range(1, self.max_steps + 1):
            print(f"\n--- Step {step} ---")

            # 1) Think: 다음 행동 결정
            action = self.think(goal)
            print(f"Thought: {action.thought}")
            print(f"Action: {action.tool}({action.input[:30]}...)")

            # 2) 종료 선언: finish 액션이면 루프 즉시 탈출
            if action.tool == "finish":
                print(f"\n에이전트가 작업 완료를 선언했습니다.")
                break

            # 3) Act: 행동 실행 및 환경 피드백 수집
            observation = self.act(action)
            print(f"Observation: {observation.result[:50]}...")

            # 4) Memory: 기록 저장 (다음 Think 입력으로 재사용)
            self.memory.append({
                'step': step,
                'action': f"{action.tool}({action.input[:20]})",
                'result': observation.result,
                'success': observation.success
            })

            # 5) Evaluate: 목표 달성 여부 자체 평가 → 달성 시 조기 종료
            if self.is_goal_achieved(goal):
                print(f"\n목표 달성!")
                break

        return {
            'goal': goal,
            'steps_taken': len(self.memory),
            'achieved': self.is_goal_achieved(goal),
            'history': self.memory
        }

In [ ]:
# 예시: 정보 수집 자율 에이전트

# 에이전트가 사용할 수 있는 도구 정의 (외부 세계와의 인터페이스)
def search_tool(query: str) -> str:
    return f"검색 결과: '{query}'에 대한 10개 문서 발견"

def analyze_tool(data: str) -> str:
    return f"분석 결과: 핵심 인사이트 3개 도출, 실행 계획 수립 완료"

def execute_tool(plan: str) -> str:
    return f"실행 결과: 계획에 따라 작업 완료, 결과물 생성됨"

def read_file_tool(path: str) -> str:
    return f"파일 내용: {path} 파일을 성공적으로 읽었습니다"

def write_file_tool(content: str) -> str:
    return f"파일 저장: 콘텐츠가 성공적으로 저장되었습니다"

# 에이전트 설정: max_steps로 폭주 방지
agent = AutonomousAgent(llm, max_steps=10)

# 도구 등록: 이름 + 함수 + LLM이 도구 선택에 활용할 상세 설명
agent.register_tool(
    "search", search_tool,
    "주제 관련 최신 자료를 웹/지식베이스에서 검색. 입력 쿼리는 2~5단어 권장."
)
agent.register_tool(
    "analyze", analyze_tool,
    "수집된 원문 자료에서 패턴/지표/핵심 인사이트를 추출. 입력은 요약되지 않은 원문."
)
agent.register_tool(
    "execute", execute_tool,
    "분석된 계획에 따라 실제 작업(문서 생성, 변환 등)을 실행. 입력은 단일 지시문."
)
agent.register_tool(
    "read_file", read_file_tool,
    "로컬 텍스트 파일을 읽어 문자열로 반환. 입력은 절대 경로."
)
agent.register_tool(
    "write_file", write_file_tool,
    "콘텐츠를 지정 경로에 저장(파일이 있으면 덮어씀)."
)

# 실행: 단일 목표만 주면 Think-Act-Observe 루프를 자율적으로 진행
result = agent.run(
    "2025~2026년 LLM 에이전트 프레임워크 시장 동향을 조사하여 "
    "800단어 내외의 요약 보고서를 작성하고 파일로 저장하세요. "
    "내용에는 주요 플레이어 비교, 오픈소스 vs 상용 추세, "
    "향후 6개월 관전 포인트가 포함되어야 합니다."
)

print(f"\n== 최종 결과 ==")
print(f"   목표 달성: {result['achieved']}")
print(f"   스텝 수: {result['steps_taken']}")
print(f"   이력 항목 수: {len(result['history'])}")


Autonomous Agent 시작
   목표: 2025~2026년 LLM 에이전트 프레임워크 시장 동향을 조사하여 800단어 내외의 요약 보고서를 작성하고 파일로 저장하세요. 내용에는 주요 플레이어 비교, 오픈소스 vs 상용 추세, 향후 6개월 관전 포인트가 포함되어야 합니다.
   사용 가능 도구: ['search', 'analyze', 'execute', 'read_file', 'write_file']
   최대 스텝: 10

--- Step 1 ---
Thought: 목표 달성을 위해 먼저 정보를 검색해야 합니다.
Action: search(2025~2026년 LLM 에이전트 프레임워크 시장 동...)
Observation: 검색 결과: '2025~2026년 LLM 에이전트 프레임워크 시장 동향을 조사하여 800단...

--- Step 2 ---
Thought: 검색 결과를 분석하여 다음 단계를 결정합니다.
Action: analyze(검색 결과: '2025~2026년 LLM 에이전트 프레...)
Observation: 분석 결과: 핵심 인사이트 3개 도출, 실행 계획 수립 완료...

--- Step 3 ---
Thought: 분석 결과를 바탕으로 작업을 실행합니다.
Action: execute(분석 결과 기반 실행...)
Observation: 실행 결과: 계획에 따라 작업 완료, 결과물 생성됨...

목표 달성!

== 최종 결과 ==
   목표 달성: True
   스텝 수: 3
   이력 항목 수: 3


---
## 요약: 6가지 패턴 비교

| 패턴 | 구조 | 사용 시기 | 복잡도 |
|------|------|----------|--------|
| **Prompt Chaining** | 순차적 | 작업이 명확한 순서로 분해 가능할 때 | 낮음 |
| **Routing** | 분기 | 입력 유형별 다른 처리가 필요할 때 | 낮음 |
| **Parallelization** | 병렬 | 독립적 작업을 동시 처리할 때 | 중간 |
| **Orchestrator-Workers** | 계층적 | 복잡한 작업을 동적으로 분해할 때 | 중간 |
| **Evaluator-Optimizer** | 반복 | 품질 기준 충족까지 개선할 때 | 중간 |
| **Autonomous Agent** | 자율 | 최소 입력으로 복잡한 목표 달성 시 | 높음 |

---
## OpenAI, Anthropic Claude LLM API 연동 예시

위 패턴들을 실제 LLM API와 연동하는 방법입니다.

In [ ]:
# OpenAI API 연동 예시
"""
from openai import OpenAI

class OpenAILLM:
    def __init__(self, model: str = "gpt-4"):
        self.client = OpenAI()
        self.model = model

    def chat(self, prompt: str, system: str = None) -> str:
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})

        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages
        )
        return response.choices[0].message.content

    async def chat_async(self, prompt: str, system: str = None) -> str:
        # AsyncOpenAI 사용
        pass

# 사용
# llm = OpenAILLM(model="gpt-4")
# chain = PromptChain(llm)
"""
print("OpenAI API 연동 코드 (주석 처리됨)")

OpenAI API 연동 코드 (주석 처리됨)


In [ ]:
# Anthropic Claude API 연동 예시
"""
from anthropic import Anthropic

class ClaudeLLM:
    def __init__(self, model: str = "claude-3-opus-20240229"):
        self.client = Anthropic()
        self.model = model

    def chat(self, prompt: str, system: str = None) -> str:
        response = self.client.messages.create(
            model=self.model,
            max_tokens=1024,
            system=system or "You are a helpful assistant.",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

    async def chat_async(self, prompt: str, system: str = None) -> str:
        # AsyncAnthropic 사용
        pass

# 사용
# llm = ClaudeLLM(model="claude-3-opus-20240229")
# agent = AutonomousAgent(llm)
"""
print("Anthropic Claude API 연동 코드 (주석 처리됨)")

Anthropic Claude API 연동 코드 (주석 처리됨)
